
# Roxy notebook example: Grouped composition descriptors

This notebook is a **reference implementation example** for the **grouped composition descriptor family** in Roxy.

It shows how to compute **composition features based on amino acid groups**, including:

- group fractions
- group counts
- compositional ratios
- validation and sanity checks
- class-style implementation for integration into Roxy

This is one of the most important descriptor families due to its **interpretability and utility for ML and EDA**.


In [1]:

from collections import Counter
import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "grp_1", "grp_2", "grp_3", "grp_4", "grp_5"
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
        ]
    }
)

df_demo


,sequence_id,sequence
0,grp_1,MKWVTFISLLFLFSSAYSRGVFRR
1,grp_2,GGGGGGGGGGGGGGG
2,grp_3,KRRKRRKRRKRRDDDDEE
3,grp_4,ACDEFGHIKLMNPQRSTVWY
4,grp_5,PPPPGSSSSSTTTTNNQQQ


## Constants and groups

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

AA_GROUPS = {
    "positive": set("KRH"),
    "negative": set("DE"),
    "charged": set("KRHDE"),
    "polar": set("STNQCYWHKRDE"),
    "nonpolar": set("AVLIMFGP"),
    "aromatic": set("FWYH"),
    "aliphatic": set("AVLIM"),
    "tiny": set("AGCS"),
    "small": set("AGCSTVPDN"),
    "branched": set("VILT"),
    "sulfur": set("CM"),
    "hydroxyl": set("STY"),
    "amide": set("NQ"),
    "hydrophobic": set("AVLIMFWCY"),
    "hydrophilic": set("RNDQEHKST"),
    "disorder_promoting": set("ARGQSEPK"),
    "order_promoting": set("CWYFILNV"),
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def group_count(seq: str, group: set) -> int:
    return sum(aa in group for aa in seq)


def group_fraction(seq: str, group: set) -> float:
    if len(seq) == 0:
        return np.nan
    return group_count(seq, group) / len(seq)


def safe_ratio(a, b):
    if b == 0:
        return np.nan
    return a / b


## Core grouped descriptor function

In [5]:

def grouped_descriptors(seq: str) -> dict:
    seq = clean_sequence(seq)
    length = len(seq)

    counts = {name: group_count(seq, group) for name, group in AA_GROUPS.items()}
    fracs = {name: group_fraction(seq, group) for name, group in AA_GROUPS.items()}

    descriptors = {
        "grp_length": length,
    }

    for name in AA_GROUPS:
        descriptors[f"grp_{name}_count"] = counts[name]
        descriptors[f"grp_{name}_frac"] = fracs[name]

    # Ratios
    descriptors["grp_ratio_acidic_basic"] = safe_ratio(counts["negative"], counts["positive"])
    descriptors["grp_ratio_basic_acidic"] = safe_ratio(counts["positive"], counts["negative"])
    descriptors["grp_ratio_polar_nonpolar"] = safe_ratio(counts["polar"], counts["nonpolar"])
    descriptors["grp_ratio_charged_uncharged"] = safe_ratio(counts["charged"], length - counts["charged"])

    return descriptors


## Apply to dataset

In [6]:

df_grp = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(grouped_descriptors).apply(pd.Series)
    ],
    axis=1
)

df_grp.head()


,sequence_id,sequence,grp_length,grp_positive_count,grp_positive_frac,grp_negative_count,grp_negative_frac,grp_charged_count,grp_charged_frac,grp_polar_count,...,grp_hydrophilic_count,grp_hydrophilic_frac,grp_disorder_promoting_count,grp_disorder_promoting_frac,grp_order_promoting_count,grp_order_promoting_frac,grp_ratio_acidic_basic,grp_ratio_basic_acidic,grp_ratio_polar_nonpolar,grp_ratio_charged_uncharged
0,grp_1,MKWVTFISLLFLFSSAYSRGVFRR,24.0,4.0,0.166667,0.0,0.000000,4.0,0.166667,11.0,...,9.0,0.375000,10.0,0.416667,12.0,0.500000,0.000000,NaN,0.846154,0.200000
1,grp_2,GGGGGGGGGGGGGGG,15.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,15.0,1.000000,0.0,0.000000,NaN,NaN,0.000000,0.000000
2,grp_3,KRRKRRKRRKRRDDDDEE,18.0,12.0,0.666667,6.0,0.333333,18.0,1.000000,18.0,...,18.0,1.000000,14.0,0.777778,0.0,0.000000,0.500000,2.0,NaN,NaN
3,grp_4,ACDEFGHIKLMNPQRSTVWY,20.0,3.0,0.150000,2.0,0.100000,5.0,0.250000,12.0,...,9.0,0.450000,8.0,0.400000,8.0,0.400000,0.666667,1.5,1.500000,0.333333
4,grp_5,PPPPGSSSSSTTTTNNQQQ,19.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,14.0,...,14.0,0.736842,13.0,0.684211,2.0,0.105263,NaN,NaN,2.800000,0.000000


## Inspect descriptor columns

In [7]:

grp_cols = [c for c in df_grp.columns if c.startswith("grp_")]
len(grp_cols)


39

## Sanity checks

In [8]:

assert "grp_positive_frac" in df_grp.columns
assert "grp_negative_frac" in df_grp.columns
assert df_grp["grp_length"].min() > 0

print("Grouped descriptor checks passed.")


Grouped descriptor checks passed.


## Class-style implementation

In [9]:

class GroupedCompositionDescriptors:

    def transform_sequence(self, seq: str) -> dict:
        return grouped_descriptors(seq)

    def transform(self, sequences):
        return pd.DataFrame([self.transform_sequence(s) for s in sequences])


grp_transformer = GroupedCompositionDescriptors()
grp_matrix = grp_transformer.transform(df_demo["sequence"])
grp_matrix.head()


,grp_length,grp_positive_count,grp_positive_frac,grp_negative_count,grp_negative_frac,grp_charged_count,grp_charged_frac,grp_polar_count,grp_polar_frac,grp_nonpolar_count,...,grp_hydrophilic_count,grp_hydrophilic_frac,grp_disorder_promoting_count,grp_disorder_promoting_frac,grp_order_promoting_count,grp_order_promoting_frac,grp_ratio_acidic_basic,grp_ratio_basic_acidic,grp_ratio_polar_nonpolar,grp_ratio_charged_uncharged
0,24,4,0.166667,0,0.000000,4,0.166667,11,0.458333,13,...,9,0.375000,10,0.416667,12,0.500000,0.000000,NaN,0.846154,0.200000
1,15,0,0.000000,0,0.000000,0,0.000000,0,0.000000,15,...,0,0.000000,15,1.000000,0,0.000000,NaN,NaN,0.000000,0.000000
2,18,12,0.666667,6,0.333333,18,1.000000,18,1.000000,0,...,18,1.000000,14,0.777778,0,0.000000,0.500000,2.0,NaN,NaN
3,20,3,0.150000,2,0.100000,5,0.250000,12,0.600000,8,...,9,0.450000,8,0.400000,8,0.400000,0.666667,1.5,1.500000,0.333333
4,19,0,0.000000,0,0.000000,0,0.000000,14,0.736842,5,...,14,0.736842,13,0.684211,2,0.105263,NaN,NaN,2.800000,0.000000


## Optional export

In [ ]:

# df_grp.to_csv("demo_grouped_descriptors.csv", index=False)
